In [4]:
import math
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
"""Weight init and Seq2Seq Encoder from Project2"""

def init_weights_seq2seq(module):
    if type(module) == nn.Linear:
        # Xavier initialization
        torch.nn.init.xavier_uniform_(module.weight)
    elif type(module) == nn.GRU:
        # Orthogonal initialization
        for param in module._flat_weights_names:
            if 'weight' in param:
                torch.nn.init.xavier_uniform_(module._parameters[param])

class Seq2SeqEncoder(d2l.Encoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers, dropout=dropout)
        self.apply(init_weights_seq2seq)

    def forward(self, X, *args):
        # X shape: (batch_size, num_steps)
        X = self.embedding(X.t().type(torch.int64))
        # X shape: (num_steps, batch_size, embed_size)
        outputs, state = self.rnn(X)
        # state shape: (num_layers, batch_size, num_hiddens)
        # output shape: (num_steps, batch_size, num_hiddens)
        return outputs, state

In [11]:
"""Test"""
vocab_size, embed_size, num_hiddens, num_layers = 10, 8, 16, 2
batch_size, num_steps = 4, 9
encoder = Seq2SeqEncoder(vocab_size, embed_size, num_hiddens, num_layers)
X = torch.zeros((batch_size, num_steps))
enc_outputs, enc_state = encoder(X)
d2l.check_shape(enc_outputs, (num_steps, batch_size, num_hiddens))


TypeError: Seq2SeqEncoder.forward() missing 1 required positional argument: 'state'